# Chapter 4 · Dynamic Programming
### Reinforcement Learning: An Introduction — Sutton & Barto (2nd edition)
An original, executable companion lesson. Follow the cells in order. We assume familiarity with finite MDPs and Bellman equations from Chapter 3.

**Learning path:** 4.1 Policy evaluation → 4.2 Policy improvement → 4.3 Policy iteration → 4.4 Value iteration → 4.5 Asynchronous DP → 4.6 Generalized policy iteration → 4.7 Efficiency.

You will derive equations, inspect every action, watch values propagate, and compare algorithms. The gridworlds are original teaching examples, not reproductions of the book's figures. DP assumes a known transition/reward model; it is planning, not learning from sampled experience.

Reference: [Sutton & Barto, book website](http://incompleteideas.net/book/the-book-2nd.html). Run locally or in Colab with NumPy and Matplotlib (`%pip install numpy matplotlib` if necessary).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
np.set_printoptions(precision=4, suppress=True)
GAMMA = .9
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ARROWS = ['↑','↓','←','→']


## Environment: a 3 × 4 gridworld
Normal moves and blocked moves cost −1. Entering goal gives +10; entering trap gives −10 instead. Terminal values are zero; the reward is paid on entry. Four deterministic actions; hitting a wall leaves the state unchanged. Discount $\gamma=0.9$.

The model stores $P(s'|s,a)$ and expected immediate reward $R(s,a)$. We use a general matrix representation so the algorithms also support stochastic transitions.

In [2]:
def make_grid(rows, cols, terminals, walls=()):
    cells = [(r,c) for r in range(rows) for c in range(cols) if (r,c) not in walls]
    idx = {s:i for i,s in enumerate(cells)}
    active = np.array([idx[s] for s in cells if s not in terminals])
    P = np.zeros((len(cells),4,len(cells)))
    R = np.zeros((len(cells),4))
    for s,i in idx.items():
        for a,(dr,dc) in enumerate(ACTIONS):
            if s in terminals:
                P[i,a,i] = 1 # absorbing, no further reward
                continue
            t = (s[0]+dr,s[1]+dc)
            if t not in idx: t=s
            P[i,a,idx[t]]=1
            R[i,a]=terminals.get(t,-1)
    return dict(rows=rows,cols=cols,cells=cells,idx=idx,active=active,P=P,R=R,terminals=terminals,walls=walls)

def uniform(m):
    p=np.zeros_like(m['R']); p[m['active']]=.25; return p

def q_values(m,v,gamma=GAMMA):
    return m['R']+gamma*(m['P']@v)

def draw(m,v=None,p=None,ax=None,title=''):
    if ax is None: _,ax=plt.subplots(figsize=(6,4))
    grid=np.full((m['rows'],m['cols']),np.nan)
    for s,i in m['idx'].items(): grid[s]=0 if v is None else v[i]
    lim=max(1,np.nanmax(np.abs(grid)))
    ax.imshow(grid,cmap='RdYlGn',vmin=-lim,vmax=lim)
    for r in range(m['rows']):
        for c in range(m['cols']):
            s=(r,c)
            if s in m['walls']: label='WALL'
            elif s in m['terminals']: label=f"{'GOAL' if m['terminals'][s]>0 else 'TRAP'}\n{m['terminals'][s]:+g} entry\nV=0"
            else:
                i=m['idx'][s]; label=f'{chr(65+r*m["cols"]+c)}'
                if v is not None: label+=f'\n{v[i]:.2f}'
                if p is not None: label+='\n'+''.join(ARROWS[a] for a in range(4) if p[i,a]>0)
            ax.text(c,r,label,ha='center',va='center',fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    ax.set_xticks(np.arange(-.5,m['cols'],1),minor=True); ax.set_yticks(np.arange(-.5,m['rows'],1),minor=True)
    ax.grid(which='minor',color='white',linewidth=2); ax.tick_params(which='minor',length=0)
    return ax

m=make_grid(3,4,{(0,3):10,(1,3):-10},[(1,1)])
# Labels are row-major letters, including positions occupied by walls/terminals.
draw(m,np.zeros(len(m['cells'])),uniform(m),title='Initial random policy: 25% each'); plt.show()
assert np.allclose(m['P'].sum(axis=-1),1)


/tmp/ipykernel_429/2312967383.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  draw(m,np.zeros(len(m['cells'])),uniform(m),title='Initial random policy: 25% each'); plt.show()


## 4.1 Policy evaluation (prediction)
For fixed $\pi$, the Bellman expectation equation is
$$v_\pi(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)[r+\gamma v_\pi(s')].$$
Initialize $v_0=0$ and use synchronous sweeps:
$$v_{k+1}=r^\pi+\gamma P^\pi v_k.$$
The policy never changes during evaluation. For the cell C immediately left of the goal, the four first-sweep targets are −1, −1, −1, +10; their average is 1.75. For the cell below C (label G in our row-major notation), the average is −3.25. Terminal states stay zero.

At convergence, $(I-\gamma P^\pi)v_\pi=r^\pi$. The linear solve gives an independent check.

In [3]:
def evaluate(m,p,gamma=GAMMA,tol=1e-10,v0=None):
    v=np.zeros(len(m['cells'])) if v0 is None else v0.copy()
    history=[v.copy()]
    for _ in range(100000):
        u=np.sum(p*q_values(m,v,gamma),axis=1)
        history.append(u.copy())
        if np.max(abs(u-v))<tol: return u,np.array(history)
        v=u
    raise RuntimeError('Evaluation did not converge')

def exact_evaluate(m,p,gamma=GAMMA):
    Ppi=np.einsum('sa,sat->st',p,m['P'])
    rpi=np.sum(p*m['R'],axis=1)
    return np.linalg.solve(np.eye(len(p))-gamma*Ppi,rpi)

p0=uniform(m)
v0,h=evaluate(m,p0)
assert np.allclose(v0,exact_evaluate(m,p0),atol=1e-8)
assert np.isclose(h[1,m['idx'][(0,2)]],1.75)
fig,axes=plt.subplots(1,4,figsize=(15,3.4))
for ax,k in zip(axes,[0,1,2,len(h)-1]): draw(m,h[k],p0,ax,f'Sweep {k}')
plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4))
for s in [(0,0),(0,2),(1,2),(2,0)]: plt.plot(h[:,m['idx'][s]],label=str(s))
plt.xlabel('Evaluation sweep');plt.ylabel('State value');plt.legend();plt.grid(alpha=.2);plt.show()
print('Sweeps:',len(h)-1,'; exact agreement:',np.max(abs(v0-exact_evaluate(m,p0))))

Sweeps: 164 ; exact agreement: 5.847233808253804e-10


/tmp/ipykernel_429/248852085.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
/tmp/ipykernel_429/248852085.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('Evaluation sweep');plt.ylabel('State value');plt.legend();plt.grid(alpha=.2);plt.show()


### Hand calculation: 2 × 2 grid
Use the familiar grid A B / C Goal, all rewards −1, including entry into goal. The random policy gives
$$v(A)=-1+.45v(A)+.225v(B)+.225v(C)$$
$$v(B)=-1+.225v(A)+.45v(B)$$
$$v(C)=-1+.225v(A)+.45v(C).$$
Moving unknowns to the left gives the coefficient matrix below. Here the nonterminal transition submatrix has rows summing to less than one when terminal transitions are omitted.

In [4]:
M=np.array([[.55,-.225,-.225],[-.225,.55,0],[-.225,0,.55]])
v_small=np.linalg.solve(M,-np.ones(3))
print(M, '\nSolution:',v_small)
small=make_grid(2,2,{(1,1):-1})
assert np.allclose(v_small,exact_evaluate(small,uniform(small))[:3])

[[ 0.55  -0.225 -0.225]
 [-0.225  0.55   0.   ]
 [-0.225  0.     0.55 ]] 
Solution: [-4.9689 -3.8509 -3.8509]


## 4.2 Policy improvement
Compute a one-step deviation from the old policy:
$$q_\pi(s,a)=\sum_{s',r}p(s',r|s,a)[r+\gamma v_\pi(s')].$$
Choose $\pi'(s)\in\arg\max_a q_\pi(s,a)$. Since a maximum is at least the policy-weighted average, $q_\pi(s,\pi'(s))\ge v_\pi(s)$. The policy improvement theorem implies $v_{\pi'}(s)\ge v_\pi(s)$ in every state. This is not the claim that $q_\pi(s,\pi'(s))=v_{\pi'}(s)$: the former follows the old policy after one action; the latter follows the new policy throughout.

We keep the existing deterministic action if it is tied for best, preventing pointless policy oscillation.

In [5]:
def improve(m,v,p,gamma=GAMMA):
    q=q_values(m,v,gamma); new=np.zeros_like(p)
    for s in m['active']:
        best=np.argmax(q[s]); old=np.argmax(p[s])
        if p[s,old]==1 and np.isclose(q[s,old],q[s,best],atol=1e-8,rtol=0): best=old
        new[s,best]=1
    return new,np.array_equal(new,p)

p1,_=improve(m,v0,p0)
s=m['idx'][(1,2)]
for a in range(4):
    t=np.argmax(m['P'][s,a]); print(f'{ARROWS[a]} -> {m["cells"][t]}, reward={m["R"][s,a]:g}, Q={q_values(m,v0)[s,a]:.5f}')
v1,_=evaluate(m,p1)
assert np.all(v1>=v0-1e-8)
fig,ax=plt.subplots(1,2,figsize=(10,4));draw(m,v0,p0,ax[0],'Old policy evaluated');draw(m,v1,p1,ax[1],'Improved policy evaluated');plt.show()

↑ -> (0, 2), reward=-1, Q=-2.27727
↓ -> (2, 2), reward=-1, Q=-8.89694
← -> (1, 2), reward=-1, Q=-7.43767
→ -> (1, 3), reward=-10, Q=-10.00000


/tmp/ipykernel_429/3230388564.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(1,2,figsize=(10,4));draw(m,v0,p0,ax[0],'Old policy evaluated');draw(m,v1,p1,ax[1],'Improved policy evaluated');plt.show()


## 4.3 Policy iteration
Outer loop: evaluate a policy to convergence, then improve it. Inner loop: repeated evaluation sweeps with fixed policy. Stop when the policy is stable. With exact evaluation and consistent tie handling, this yields an optimal policy for a finite discounted MDP. Numerical evaluation uses a tolerance.

$$\pi_0\xrightarrow{E}v_{\pi_0}\xrightarrow{I}\pi_1\xrightarrow{E}v_{\pi_1}\xrightarrow{I}\cdots$$

In [6]:
def policy_iteration(m,gamma=GAMMA,p=None):
    p=uniform(m) if p is None else p.copy(); records=[]
    for i in range(1000):
        v,h=evaluate(m,p,gamma)
        new,stable=improve(m,v,p,gamma)
        records.append(dict(p=p.copy(),v=v.copy(),h=h,changed=np.count_nonzero(np.any(new!=p,axis=1))))
        if stable:return p,v,records
        p=new
    raise RuntimeError('Policy iteration limit')

pstar,vstar,records=policy_iteration(m)
fig,axes=plt.subplots(1,len(records),figsize=(5*len(records),4),squeeze=False)
for i,rec in enumerate(records):
    draw(m,rec['v'],rec['p'],axes[0,i],f'Round {i} | changes: {rec["changed"]}')
    print(f'Round {i}: {len(rec["h"])-1} evaluation sweeps; {rec["changed"]} changed states')
plt.tight_layout();plt.show()
for a,b in zip(records,records[1:]):assert np.all(b['v']>=a['v']-1e-8)
assert np.max(abs(vstar[m['active']]-q_values(m,vstar)[m['active']].max(axis=1)))<1e-8


Round 0: 164 evaluation sweeps; 9 changed states
Round 1: 6 evaluation sweeps; 0 changed states


/tmp/ipykernel_429/3514695369.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


### Why more than one round may be necessary
A deliberately poor initial policy (all states choose up) creates loops. Its evaluation assigns negative values to those loops. Improvement discovers better directions in successive rounds. This second experiment makes the outer loop particularly visible.

In [7]:
poor=np.zeros_like(p0);poor[m['active'],0]=1
_,_,long_records=policy_iteration(m,p=poor)
fig,axes=plt.subplots(1,len(long_records),figsize=(4*len(long_records),3.5),squeeze=False)
for i,rec in enumerate(long_records):draw(m,rec['v'],rec['p'],axes[0,i],f'Round {i} | Δπ={rec["changed"]}')
plt.tight_layout();plt.show()
assert np.allclose(long_records[-1]['v'],vstar,atol=1e-8)

/tmp/ipykernel_429/4017688822.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


## 4.4 Value iteration
Instead of full evaluation followed by improvement, use a Bellman optimality backup each sweep:
$$v_{k+1}(s)=\max_a\sum_{s',r}p(s',r|s,a)[r+\gamma v_k(s')].$$
Extract a greedy policy after convergence. Intermediate values are not necessarily the fully evaluated value of any policy. The maximum replaces the policy-weighted average.

In [8]:
def value_iteration(m,gamma=GAMMA,tol=1e-10):
    v=np.zeros(len(m['cells']));h=[v.copy()]
    for _ in range(100000):
        u=q_values(m,v,gamma).max(axis=1);h.append(u.copy())
        if np.max(abs(u-v))<tol:return improve(m,u,uniform(m),gamma)[0],u,np.array(h)
        v=u
    raise RuntimeError('VI limit')
pvi,vvi,hvi=value_iteration(m)
assert np.allclose(vvi,vstar,atol=1e-8)
fig,axes=plt.subplots(1,4,figsize=(15,3.5))
for ax,k in zip(axes,[0,1,2,len(hvi)-1]):draw(m,hvi[k],None,ax,f'VI sweep {k}')
plt.tight_layout();plt.show()
print('PI and VI agree on values:',np.allclose(vvi,vstar))
print('Policies may differ on tied actions.')

PI and VI agree on values: True
Policies may differ on tied actions.


/tmp/ipykernel_429/3516507886.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


## 4.5 Asynchronous dynamic programming
Synchronous sweeps use a frozen copy of the previous vector. Asynchronous DP updates selected states in place, possibly in a different order. A cyclic in-place optimality update is one example. For finite discounted MDPs, repeated updates of every state converge to the optimal values. Update order influences the speed, not the solution.

In [9]:
def async_vi(m,order,gamma=GAMMA,tol=1e-10):
    v=np.zeros(len(m['cells']));hist=[v.copy()]
    for _ in range(100000):
        old=v.copy()
        for s in order: v[s]=np.max(m['R'][s]+gamma*(m['P'][s]@v))
        hist.append(v.copy())
        if np.max(abs(v-old))<tol:return v,np.array(hist)
    raise RuntimeError('Async limit')
va,ha=async_vi(m,m['active']); vb,hb=async_vi(m,m['active'][::-1])
assert np.allclose(va,vstar,atol=1e-8) and np.allclose(vb,vstar,atol=1e-8)
for label,hist in [('Synchronous',hvi),('In-place forward',ha),('In-place reverse',hb)]:
    err=np.max(abs(hist-vstar),axis=1);plt.semilogy(np.maximum(err,1e-12),marker='.',label=label)
plt.xlabel('Full passes');plt.ylabel('Max absolute error');plt.legend();plt.grid(alpha=.2);plt.show()

/tmp/ipykernel_429/821889466.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('Full passes');plt.ylabel('Max absolute error');plt.legend();plt.grid(alpha=.2);plt.show()


## 4.6 Generalized policy iteration (GPI)
Evaluation and improvement need not each run to completion before the other begins. GPI describes their interaction: evaluation pushes values toward consistency with the current policy; improvement pushes the policy toward greediness with respect to current values. Modified policy iteration illustrates this by using a limited number of expectation sweeps before greedification. One sweep is not automatically identical to value iteration from an arbitrary initial policy; the timing of greedification matters.

The following experiment starts with a random policy and warm-starts the value vector between rounds. We stop on a Bellman optimality residual, not just unchanged policy, because evaluation is truncated.

In [10]:
def modified_pi(m,sweeps,gamma=GAMMA,tol=1e-9):
    p=uniform(m);v=np.zeros(len(p)); errors=[]
    for _ in range(10000):
        for k in range(sweeps):v=np.sum(p*q_values(m,v,gamma),axis=1)
        p,_=improve(m,v,p,gamma)
        residual=np.max(abs(q_values(m,v,gamma).max(axis=1)-v));errors.append(residual)
        if residual<tol:return v,errors
    raise RuntimeError('MPI limit')
for k in [1,3,10]:
    v,err=modified_pi(m,k);assert np.allclose(v,vstar,atol=1e-7)
    plt.semilogy(np.maximum(err,1e-12),label=f'{k} eval sweeps per round')
plt.xlabel('Outer round');plt.ylabel('Optimality residual');plt.legend();plt.grid(alpha=.2);plt.show()

/tmp/ipykernel_429/421174646.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('Outer round');plt.ylabel('Optimality residual');plt.legend();plt.grid(alpha=.2);plt.show()


## 4.7 Efficiency of dynamic programming
For dense transitions a full backup is $O(|S|^2|A|)$; deterministic or sparse models can avoid summing over unreachable next states. A dense direct evaluation solve generally costs $O(|S|^3)$ arithmetic, whereas repeated sweeps exploit sparsity. DP can still be expensive when the state space explodes. The comparison below counts state-action lookaheads, not wall-clock time; the implementation uses all actions even when the policy is deterministic. A lookahead itself can involve several successor states.

In [11]:
nstate=len(m['active']);na=4
pi_work=sum((len(r['h'])-1)+1 for r in records)*nstate*na
vi_work=((len(hvi)-1)+1)*nstate*na
plt.bar(['Policy iteration','Value iteration'],[pi_work,vi_work]);plt.ylabel('State-action lookaheads');plt.title('Illustrative work count on this grid');plt.show()
print('PI:',pi_work,'VI:',vi_work)

PI: 6192 VI: 252


/tmp/ipykernel_429/2853877170.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.bar(['Policy iteration','Value iteration'],[pi_work,vi_work]);plt.ylabel('State-action lookaheads');plt.title('Illustrative work count on this grid');plt.show()


## Extension: stochastic actions
The same Bellman equations work when an intended action can slip. Here the intended direction occurs with probability 0.8 and each perpendicular direction with probability 0.1. The expected reward and successor distribution must both be updated. Risk near the trap can change the optimal route.

In [12]:
slip=dict(m);P=m['P'].copy();R=m['R'].copy()
for a in range(4):
    perp=[2,3] if a<2 else [0,1]
    P[:,a]=.8*m['P'][:,a]+.1*m['P'][:,perp[0]]+.1*m['P'][:,perp[1]]
    R[:,a]=.8*m['R'][:,a]+.1*m['R'][:,perp[0]]+.1*m['R'][:,perp[1]]
slip.update(P=P,R=R)
ps,vs,_=policy_iteration(slip)
_,check,_=value_iteration(slip);assert np.allclose(vs,check,atol=1e-8)
fig,ax=plt.subplots(1,2,figsize=(10,4));draw(m,vstar,pstar,ax[0],'Deterministic');draw(slip,vs,ps,ax[1],'Stochastic actions');plt.show()

/tmp/ipykernel_429/553566914.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(1,2,figsize=(10,4));draw(m,vstar,pstar,ax[0],'Deterministic');draw(slip,vs,ps,ax[1],'Stochastic actions');plt.show()


## Exercises (try before reading solutions)
1. Derive the first two evaluation sweeps for the cell immediately left of the goal.
2. Explain why the terminal value is zero even though entering the goal earns +10.
3. Change gamma to 0.5 and 0.99. Compare the optimal values.
4. Start policy iteration with all actions pointing left. Does the final value change?
5. Why is a stable policy alone insufficient when policy evaluation is truncated?
6. Agentic RAG: define states, actions, transition model, reward and terminal condition for a retrieve/rewrite/answer router. Explain when DP is feasible.

In [13]:
for g in [.5,.9,.99]:
    p,v,_=policy_iteration(m,g);print('gamma',g,'start value',v[m['idx'][(2,0)]])
left=np.zeros_like(p0);left[m['active'],2]=1
_,vleft,_=policy_iteration(m,p=left)
assert np.allclose(vleft,vstar,atol=1e-8)

gamma 0.5 start value -1.25
gamma 0.9 start value 3.122
gamma 0.99 start value 5.665561100000001


### Solutions and checklist
1. $v_1(C)=1.75$; $v_2(C)=(0.575-3.925-1.9+10)/4=1.1875$.
2. Entry reward belongs to the transition; no further reward is earned after termination.
3. See the printed values above. Discounting changes the valuation of later rewards.
4. No, the optimal value is unique in a finite discounted MDP; optimal policies can differ on ties.
5. A temporarily stable greedy policy with inaccurate values may change after further evaluation. Check an optimality residual or fully evaluate before testing stability.
6. Example: state = question type, evidence quality, remaining budget; actions = retrieve, rewrite, answer; reward = terminal answer quality minus tool cost; terminal = answer or budget exhaustion. DP requires known transition probabilities and rewards, so this is feasible for a small estimated model; real RAG often needs model learning or other RL methods and may be partially observable.

**Remember:** evaluation = average under fixed policy; improvement = greedy choice based on old values; policy iteration = alternate full evaluation and improvement; value iteration = max backups; asynchronous DP = flexible update order; GPI = the general interaction of evaluation and improvement.
